let’s build an LSTM-based text generator from scratch using PyTorch, but without using nn.LSTM. We'll manually implement the LSTM cell logic, but let PyTorch handle autograd and optimization.

In [ ]:
import xml.etree.ElementTree as ET

def extract_wikipedia_text(xml_path, max_pages=None):
    """Extracts plain text from <text> tags in the Wikipedia XML dump."""
    extracted_text = []
    page_count = 0

    for event, elem in ET.iterparse(xml_path, events=("start", "end")):
        if event == "end" and elem.tag.endswith("text"):
            text_content = elem.text
            if text_content:
                # Basic cleanup: remove things like [[link|display]], {{templates}}, and ==
                cleaned = (
                    text_content
                    .replace('\n', ' ')  # remove newlines
                    .replace('[[', '').replace(']]', '')
                )
                extracted_text.append(cleaned)

                page_count += 1
                if max_pages and page_count >= max_pages:
                    break
            elem.clear()

    return " ".join(extracted_text)

In [ ]:
# Set the path to your Simple English Wikipedia XML file
wiki_path = r"/content/simplewiki-latest-pages-articles.xml"

# Extract text (you can set max_pages to limit processing time)
simplewiki_text = extract_wikipedia_text(wiki_path, max_pages=1000)

# View sample
print("Sample text:\n", simplewiki_text[:1000])

Sample text:
 {{monththisyear|4}} '''April''' (Apr.) is the fourth month of the year in the Julian calendar|Julian and Gregorian calendars, and comes between March and May. It is one of four months to have 30 days.  April always begins on the same day of the week as July, and additionally, January in leap years. April always ends on the same day of the week as December.  == The Month == File:Colorful spring garden.jpg|thumb|180px|right|Spring (season)|Spring flowers in April in the Northern Hemisphere. April comes between March and May, making it the fourth month of the year. It also comes first in the year out of the four months that have 30 days, as June, September and November are later in the year.  April begins on the same day of the week as July every year and on the same day of the week as January in leap years. April ends on the same day of the week as December every year, as each other's last days are exactly 35 weeks (245 days) apart.  In common years, April starts on the sam

In [ ]:
# Assuming `simplewiki_text` already holds the extracted Wikipedia text
# CLEANING STEP
import re
clean_text = re.sub(r'[^\x00-\x7F]+', ' ', simplewiki_text)  # Remove non-ASCII chars
clean_text = re.sub(r'={2,}.*?={2,}', ' ', clean_text)       # Remove headers like "== History =="
clean_text = re.sub(r'\s+', ' ', clean_text).strip()         # Normalize whitespace

In [ ]:
with open("simplewiki_cleaned.txt", "w", encoding="utf-8") as f:
    f.write(clean_text)
import re
clean_text = re.sub(r'[^\w\s]', '', clean_text)  # Keep only words and spaces
clean_text = re.sub(r'\d+', '', clean_text)      # Remove numbers


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from collections import Counter
import torch.nn.functional as F  # Make sure this is imported at the top
from collections import defaultdict, Counter
 # -----------------------------------
# Preprocessing
# -----------------------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'

words = clean_text.lower().split()
counts = Counter(words)
most_common = [w for w, _ in counts.most_common(10000)]
vocab = ['<UNK>'] + most_common
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
vocab_size = len(vocab)

def encode(w_list):
    return [word2idx.get(w, word2idx['<UNK>']) for w in w_list]

def decode(i_list):
    return ' '.join([idx2word.get(i, "<UNK>") for i in i_list])

data = torch.tensor(encode(words), dtype=torch.long).to(device)

# -----------------------------------
# LSTM Cell with Swish Activation
# -----------------------------------
class LSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.W = nn.Parameter(torch.randn(4 * hidden_size, input_size) * 0.1)
        self.U = nn.Parameter(torch.randn(4 * hidden_size, hidden_size) * 0.1)
        self.b = nn.Parameter(torch.zeros(4 * hidden_size))

    def forward(self, x, h, c):
        gates = self.W @ x + self.U @ h + self.b
        i, f, o, g = torch.chunk(gates, 4, dim=0)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = F.silu(g)  # Swish activation
        c_next = f * c + i * g
        h_next = o * F.silu(c_next)  # Optional Swish on output
        return h_next, c_next

# -----------------------------------
# Word-Level LSTM Model
# -----------------------------------
class WordLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = LSTMCell(embedding_dim, hidden_size)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h, c):
        logits = []
        for t in range(x.size(0)):
            emb = self.embedding(x[t])
            h, c = self.lstm(emb.view(-1), h, c)
            out = self.fc(h)
            logits.append(out)
        return torch.stack(logits), h, c

# -----------------------------------
# Training
# -----------------------------------
embedding_dim = 128
hidden_size = 512
seq_len = 10
lr = 0.001
num_epochs = 1000

model = WordLSTM(vocab_size, embedding_dim, hidden_size).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
loss_fn = nn.CrossEntropyLoss()
grpo_memory = []  # Initialize GRPO memory

for epoch in range(num_epochs):
    i = np.random.randint(0, len(data) - seq_len - 1)
    seq = data[i:i+seq_len]
    target = data[i+1:i+seq_len+1]

    h = torch.zeros(hidden_size).to(device)
    c = torch.zeros(hidden_size).to(device)

    logits, h, c = model(seq, h, c)
    loss = loss_fn(logits.view(-1, vocab_size), target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # -----------------------------------
    # Create GRPO Memory (During Training)
    # -----------------------------------
    with torch.no_grad():
        predicted_indices = torch.argmax(logits, dim=-1)
        grpo_memory.append((seq.tolist(), predicted_indices.tolist()))

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

# -----------------------------------
# Generation
# -----------------------------------
def generate(seed_text, max_len=20):
    model.eval()
    seed_words = seed_text.lower().split()
    input_ids = torch.tensor(encode(seed_words), dtype=torch.long).to(device)
    h = torch.zeros(hidden_size).to(device)
    c = torch.zeros(hidden_size).to(device)
    output = seed_words.copy()

    for i in range(len(seed_words) - 1):
        _, h, c = model(input_ids[i:i+1], h, c)

    next_word = input_ids[-1]
    for _ in range(max_len):
        logits, h, c = model(next_word.view(1), h, c)
        probs = F.softmax(logits[-1], dim=-1)
        next_word = torch.multinomial(probs, 1).item()
        output.append(idx2word.get(next_word, "<UNK>"))
        next_word = torch.tensor(next_word).to(device)

    return ' '.join(output)

# -----------------------------------
# Output
# -----------------------------------
print("\nGenerated text:")
print(generate("AI can change"))


Epoch 0, Loss: 9.1588
Epoch 50, Loss: 7.5866
Epoch 100, Loss: 8.0282
Epoch 150, Loss: 7.7239
Epoch 200, Loss: 9.4131
Epoch 250, Loss: 7.8718
Epoch 300, Loss: 6.5370
Epoch 350, Loss: 7.7328
Epoch 400, Loss: 7.9294
Epoch 450, Loss: 9.2289
Epoch 500, Loss: 7.7760
Epoch 550, Loss: 7.2935
Epoch 600, Loss: 5.8915
Epoch 650, Loss: 6.8924
Epoch 700, Loss: 6.4831
Epoch 750, Loss: 7.1585
Epoch 800, Loss: 7.4178
Epoch 850, Loss: 6.8155
Epoch 900, Loss: 5.9908
Epoch 950, Loss: 7.0539

Generated text:
ai can change and mode netherland <UNK> <UNK> vikings galileo gini on the country started to you <UNK> <UNK> how speedwords archivedate and
